# Compile ph and summary data from infotaxis runs with restricted beam movements

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

In [2]:
path_main = Path("../simulations")

# Path to save output csv
path_csv = path_main / "20251118_summary_refined"
if not path_csv.exists():
    path_csv.mkdir(parents=True, exist_ok=True)

In [3]:
def load_summary(path_data, search_type, cr, br, bmr, pm):
    return pd.read_csv(
        path_data / f"{search_type}_cr{cr}_br{br}_bmr{bmr}_pm{pm:.0e}_pfa{pm:.0e}_summary.csv",
        index_col=0
    )

In [4]:
def load_ph(path_data, search_type, cr, br, bmr, pm):
    return xr.open_dataset(
        path_data / f"{search_type}_cr{cr}_br{br}_bmr{bmr}_pm{pm:.0e}_pfa{pm:.0e}_ph.nc"
    )

In [5]:
def get_ping_cross_p_max_th(ds, p_max_th=0.95):    
    ping_cross_all = []
    for run in ds["run"].values:
        ping_cross_th = ds["p_max"].sel(run=run).dropna(dim="ping").values > p_max_th
        if ping_cross_th.sum() > 0:
            ping_cross = np.argwhere(ping_cross_th).min()
        else:
            ping_cross = 999
        ping_cross_all.append(int(ping_cross))
    return np.array(ping_cross_all)

In [6]:
def get_ping_p_max_diff_th(ds, pmax_diff_threshold=1e-5):
    criteria = {
        "pmax_repeat_N": 3,
        "max_ping_num": 500,
        "pmax_diff_threshold": pmax_diff_threshold,
    }    
    p_all = []
    for run in ds["run"].values:
        p_max = ds["p_max"].sel(run=run).dropna(dim="ping")
        for p in range(len(p_max)):
            if p - criteria["pmax_repeat_N"] < 0:
                continue
            p_max_diff = np.diff(p_max[p-criteria["pmax_repeat_N"]+1:p+1])
            if len(p_max_diff) > 1 and np.all(abs(p_max_diff) < criteria["pmax_diff_threshold"]):
                p_all.append(p)
                break
    return np.array(p_all)

In [7]:
def refine_num_pings(cr, br, bmr, pm_all):
    for idx, pm in enumerate(pm_all):
        print("------------------------------------------------")
        print(f"cr={cr}, br={br}, bmr={bmr}, pm={pm}, pfa={pm}")

        # Load summary and ph details
        df_y = load_summary(path_main / "20251118_summary", "infotaxis", cr, br, bmr, pm)
        # Get num_pings when p_max crosses threshold
        ds_y = load_ph(path_main / "20251118_summary", "infotaxis", cr, br, bmr, pm)
        y1 = get_ping_cross_p_max_th(ds_y)
        y2 = get_ping_p_max_diff_th(ds_y, pmax_diff_threshold=1e-5)

        # Substitute num_pings in the runs that did not produce pmax > threshold
        # with pmax convergence num_pings
        idx_no_cross_y = y1==999
        y = y1.copy()
        y[idx_no_cross_y] = y2[idx_no_cross_y]

        # Store refined number of pings into the summary df
        df_y["num_pings_cross_pmax_th"] = y1
        df_y["num_pings_pmax_diff"] = y2
        df_y["num_pings_2conditions"] = y

        fname_postfix = f"cr{cr}_br{br}_bmr{bmr}_pm{pm:.0e}_pfa{pm:.0e}_summary.csv"
        df_y.to_csv(path_csv / f"infotaxis_{fname_postfix}")

        print("save refined info to:")
        print(f" - infotaxis_{fname_postfix}")

In [8]:
cr_all = [5, 10]
br_all = [1, 2]
bmr_all = [2, 3]
pm_all = [0.001, 0.01, 0.02, 0.05]

In [9]:
for cr in cr_all:
    for br in br_all:
        for bmr in bmr_all:
            refine_num_pings(cr, br, bmr, pm_all)

------------------------------------------------
cr=5, br=1, bmr=2, pm=0.001, pfa=0.001
save refined info to:
 - infotaxis_cr5_br1_bmr2_pm1e-03_pfa1e-03_summary.csv
------------------------------------------------
cr=5, br=1, bmr=2, pm=0.01, pfa=0.01
save refined info to:
 - infotaxis_cr5_br1_bmr2_pm1e-02_pfa1e-02_summary.csv
------------------------------------------------
cr=5, br=1, bmr=2, pm=0.02, pfa=0.02
save refined info to:
 - infotaxis_cr5_br1_bmr2_pm2e-02_pfa2e-02_summary.csv
------------------------------------------------
cr=5, br=1, bmr=2, pm=0.05, pfa=0.05
save refined info to:
 - infotaxis_cr5_br1_bmr2_pm5e-02_pfa5e-02_summary.csv
------------------------------------------------
cr=5, br=1, bmr=3, pm=0.001, pfa=0.001
save refined info to:
 - infotaxis_cr5_br1_bmr3_pm1e-03_pfa1e-03_summary.csv
------------------------------------------------
cr=5, br=1, bmr=3, pm=0.01, pfa=0.01
save refined info to:
 - infotaxis_cr5_br1_bmr3_pm1e-02_pfa1e-02_summary.csv
------------------